# MLP：Train-fitted 标准化与 One-Hot 神经网络挑战者

本 Notebook 使用与所有前序模型相同的 126 个公共特征和固定 Train/Selection 切分。数值变量在 Train 上做中位数填补、缺失指示和标准化；类别变量在 Train 上做显式缺失类别与 One-Hot，Selection 未见类别忽略。每个网络仅在 Train 上以 `partial_fit` 更新；Selection AUC 用于 early stopping、锁模和与共同输入 Logistic 的配对比较。这样不会使用 sklearn 默认的 accuracy 内部早停。锁模后用 3,000 行 Selection 样本上的原始变量级 permutation importance 解释网络，Test 不在本 Notebook 中读取。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'configs' / 'experiment.yaml').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.comparison import load_locked_selection_prediction
from credit_risk.development import load_development_data
from credit_risk.models.mlp import run_mlp
print('Project root:', PROJECT_ROOT)

Project root: /Users/ganlu/Desktop/home-credit-risk-english


In [2]:
development = load_development_data(PROJECT_ROOT)
assert not hasattr(development, 'X_test')
assert not hasattr(development, 'y_test')
reference_selection_prediction = load_locked_selection_prediction(
    development, 'common_logistic'
)
display(development.split_overview)
print('公共特征数:', len(development.feature_columns))
print('已对齐的共同输入 Logistic Selection 预测数:', len(reference_selection_prediction))

,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test (sealed),46127,NaN


公共特征数: 126
已对齐的共同输入 Logistic Selection 预测数: 46127


In [3]:
search_settings = development.configs['model_search_spaces']['mlp']
display(pd.DataFrame(search_settings['candidate_profiles']))
print('基础 epoch 上限:', search_settings['max_epochs'])
print('边界延长上限:', search_settings['extension_max_epochs'])
print('AUC early stopping patience:', search_settings['early_stopping_rounds'])

,name,hidden_layer_sizes,alpha,learning_rate_init,batch_size
0,single_64,[64],0.001,0.0008,512
1,compact_64_32,"[64, 32]",0.001,0.0008,512
2,wider_128_64,"[128, 64]",0.001,0.0005,512
3,compact_strong_l2,"[64, 32]",0.010,0.0010,512
4,wider_strong_l2,"[128, 64]",0.010,0.0008,512
5,wide_single,[128],0.001,0.0010,512


基础 epoch 上限: 200
边界延长上限: 300
AUC early stopping patience: 20


In [4]:
result = run_mlp(
    development,
    reference_selection_prediction=reference_selection_prediction,
    register_lock=True,
)
print('MLP 搜索、固定 epoch 重训与锁模完成。')

MLP candidate 1 [base_search/single_64] Selection AR=0.495242, gap=0.007546, epoch=63/83, fit=29.5s


MLP candidate 2 [base_search/compact_64_32] Selection AR=0.495903, gap=0.007143, epoch=31/51, fit=19.7s


MLP candidate 3 [base_search/wider_128_64] Selection AR=0.496873, gap=0.009569, epoch=33/53, fit=33.3s


MLP candidate 4 [base_search/compact_strong_l2] Selection AR=0.497635, gap=0.008952, epoch=49/69, fit=26.1s


MLP candidate 5 [base_search/wider_strong_l2] Selection AR=0.495844, gap=0.004016, epoch=19/39, fit=24.3s


MLP candidate 6 [base_search/wide_single] Selection AR=0.494092, gap=0.006640, epoch=33/53, fit=30.3s


MLP 搜索、固定 epoch 重训与锁模完成。


In [5]:
display(result.search_results)
display(result.metrics)
display(result.uplift_vs_logistic)
display(result.permutation_importance.head(30))
display(result.selection_deciles)
display(result.locked_config)
print('锁模产物:', result.run_directory)

,candidate,stage,profile,max_epochs,attempted_epochs,best_iteration,reached_iteration_boundary,hidden_layer_sizes,alpha,learning_rate_init,...,selection_ar,ar_gap,train_ks,selection_ks,selection_ap,selection_logloss,capacity,fit_seconds,parameters_json,eligible_for_lock
0,4,base_search,compact_strong_l2,200,69,49,False,"[64, 32]",0.010,0.0010,...,0.497635,0.008952,0.376808,0.368092,0.235094,0.249026,10035200.0,26.120844,"{""activation"": ""relu"", ""alpha"": 0.01, ""batch_s...",True
1,3,base_search,wider_128_64,200,53,33,False,"[128, 64]",0.001,0.0005,...,0.496873,0.009569,0.378270,0.362084,0.235069,0.258641,270336000.0,33.334136,"{""activation"": ""relu"", ""alpha"": 0.001, ""batch_...",True
2,2,base_search,compact_64_32,200,51,31,False,"[64, 32]",0.001,0.0008,...,0.495903,0.007143,0.375853,0.365160,0.232783,0.255433,63488000.0,19.684046,"{""activation"": ""relu"", ""alpha"": 0.001, ""batch_...",True
3,5,base_search,wider_strong_l2,200,39,19,False,"[128, 64]",0.010,0.0008,...,0.495844,0.004016,0.372846,0.365596,0.234469,0.250054,15564800.0,24.299844,"{""activation"": ""relu"", ""alpha"": 0.01, ""batch_s...",True
4,1,base_search,single_64,200,83,63,False,[64],0.001,0.0008,...,0.495242,0.007546,0.374488,0.362356,0.234516,0.249005,4032000.0,29.483104,"{""activation"": ""relu"", ""alpha"": 0.001, ""batch_...",True
5,6,base_search,wide_single,200,53,33,False,[128],0.001,0.0010,...,0.494092,0.006640,0.371757,0.362902,0.231976,0.249745,4224000.0,30.279578,"{""activation"": ""relu"", ""alpha"": 0.001, ""batch_...",True


,sample,n,bad_rate,AUC,AR,KS,AP,Logloss
0,Train,215257,0.080727,0.749930,0.499860,0.372846,0.229583,0.249786
1,Selection,46127,0.080734,0.747922,0.495844,0.365596,0.234469,0.250054


,model,reference_model,metric,difference,lower,upper,bootstrap_se,confidence_level,bootstrap_replicates,higher_is_better
0,mlp,common_logistic,AUC,-0.000221,-0.001389,0.000838,0.000569,0.95,500,True
1,mlp,common_logistic,AR,-0.000442,-0.002777,0.001677,0.001138,0.95,500,True
2,mlp,common_logistic,KS,-0.000881,-0.005602,0.003679,0.002401,0.95,500,True
3,mlp,common_logistic,AP,-0.000961,-0.002757,0.000658,0.000872,0.95,500,True
4,mlp,common_logistic,Logloss,0.001378,0.001040,0.001709,0.000163,0.95,500,False


,feature,baseline_auc,mean_auc_decrease,std_auc_decrease,permutation_repeats,sample_size
0,EXT_SOURCE_3,0.719594,0.070068,0.011397,3,3000
1,EXT_SOURCE_2,0.719594,0.033362,0.004652,3,3000
2,EXT_SOURCE_1,0.719594,0.014736,0.002844,3,3000
3,NAME_EDUCATION_TYPE,0.719594,0.002700,0.003578,3,3000
4,PHONE_CHANGE_YEARS,0.719594,0.002535,0.000462,3,3000
5,EMPLOYED_YEARS,0.719594,0.001836,0.001553,3,3000
6,GOODS_CREDIT_RATIO,0.719594,0.001637,0.000940,3,3000
7,REG_REGION_NOT_WORK_REGION,0.719594,0.001542,0.000893,3,3000
8,ID_PUBLISH_YEARS,0.719594,0.001473,0.001461,3,3000
9,NAME_FAMILY_STATUS,0.719594,0.001451,0.000910,3,3000


,risk_group,rank_min,rank_max,n,bad_n,bad_rate,probability_min,probability_max,lift,cumulative_bad_capture
0,1,1,4613,4613,1208,0.261869,0.206949,0.699592,3.243613,0.324382
1,2,4614,9226,4613,660,0.143074,0.141151,0.206914,1.772173,0.501611
2,3,9227,13839,4613,482,0.104487,0.105823,0.141147,1.294223,0.631042
3,4,13840,18451,4612,360,0.078057,0.082166,0.105808,0.966849,0.727712
4,5,18452,23064,4613,307,0.066551,0.065315,0.082161,0.824329,0.810150
5,6,23065,27677,4613,233,0.050509,0.052680,0.065311,0.625631,0.872718
6,7,27678,32289,4612,177,0.038378,0.042063,0.052677,0.475368,0.920247
7,8,32290,36902,4613,143,0.030999,0.032869,0.042062,0.383971,0.958647
8,9,36903,41515,4613,94,0.020377,0.023812,0.032868,0.252400,0.983888
9,10,41516,46127,4612,60,0.013010,0.000262,0.023810,0.161142,1.000000


{'model_name': 'mlp',
 'experiment_track': 'controlled',
 'data_sha256': '52e96b895b1112e1c853f670e58372719c8441c5ed1c57ac2f7fad559d784f5f',
 'split_sha256': '4ae4ceee661ba8c210fa0a1cf950448799a0aef468659e6d2590c1bb4e8a3835',
 'feature_policy_version': 1,
 'training_sample': 'Train only',
 'selection_role': 'AUC early stopping, candidate selection, and paired comparison only',
 'test_accessed': False,
 'parameters': {'hidden_layer_sizes': (128, 64),
  'activation': 'relu',
  'solver': 'adam',
  'alpha': 0.01,
  'batch_size': 512,
  'learning_rate_init': 0.0008,
  'max_iter': 1,
  'shuffle': True,
  'random_state': 42,
  'early_stopping': False,
  'tol': 0.0},
 'epochs': 19,
 'selected_profile': 'wider_strong_l2',
 'candidate_count': 6,
 'base_candidate_count': 6,
 'boundary_extensions': 0,
 'selected_candidate_reached_extension_boundary': False,
 'final_refit_on_train_only_at_fixed_epoch': True,
 'raw_feature_count': 126,
 'numeric_feature_count': 111,
 'categorical_feature_count': 15,

锁模产物: /Users/ganlu/Desktop/home-credit-risk-english/artifacts/mlp/20260831T072344_519831Z


In [6]:
registry_path = PROJECT_ROOT / development.configs['experiment']['test_policy']['registry_path']
registry = json.loads(registry_path.read_text(encoding='utf-8'))
assert registry['models']['mlp']['locked'] is True
assert registry['models']['mlp']['config']['test_accessed'] is False
assert registry['suite_locked'] is True
print('全部模型已锁定；本 Notebook 未读取 Test。最终 Test 评估必须在单独的揭盲 Notebook 中一次性执行。')

全部模型已锁定；本 Notebook 未读取 Test。最终 Test 评估必须在单独的揭盲 Notebook 中一次性执行。
